# Concierge de Viajes Multiagente

> "Buscame vuelos a Bariloche, un hotel cerca del centro y que hacer el finde."

Un solo agente se marea con todo eso. Vamos a arrancar sin ningun framework, con una
clase de Python y un loop, y de ahi construimos hasta tener varios agentes que se
coordinan.

## Lo que vas a poder hacer al final

- Escribir un agente desde cero, sin framework, y entender exactamente que hace
- Saber que te da un framework cuando lo usas, porque primero lo hiciste a mano
- Ver por que un solo agente con muchas tools se equivoca, y como se arregla
- Coordinar varios agentes con un supervisor
- Entender que MCP es un protocolo de tools, nada mas

## Agenda

| Seccion | Tema | Min |
|---|---|---|
| 1 | El caso: tres pedidos en una sola frase | 3 |
| 2 | Un agente es un loop | 10 |
| 3 | Lo mismo, pero en LangGraph | 7 |
| 4 | Un agente con demasiadas tools | 7 |
| 5 | Lo partimos: un agente por capacidad | 8 |
| 6 | Quien le contesta al usuario | 9 |
| 7 | MCP, sin misticismo | 8 |
| 8 | Y si se organizan solos | 5 |
| 9 | Cierre | 3 |

Taller de Axel Sirota para Nerdearla.

**Todas las celdas corren tal como estan.** Los ejercicios son opcionales y no
bloquean nada: si te los salteas, el notebook sigue funcionando igual. Las
soluciones estan al final.


## Seccion 0: Preparamos el entorno

Dos celdas y arrancamos. La primera instala todo, la segunda carga las claves.

Axel les reparte un archivo `.env` con las claves adentro. En Colab, subilo con el
panel de la izquierda (el iconito de carpeta) y dejalo al lado del notebook. No hace
falta que toques nada mas.


In [ ]:
# Instalamos todo de una. En Colab esto tarda un minuto la primera vez.
# Las versiones estan clavadas a proposito: son las que probamos y sabemos que andan juntas.
!pip install -q \
    openai==3.16.2 \
    python-dotenv==1.2.3 \
    requests \
    langchain==1.4.2 \
    langchain-openai==1.6.3 \
    langgraph==1.2.12 \
    langgraph-supervisor==0.0.31 \
    langgraph-swarm==0.1.0 \
    mcp==2.2.0 \
    fastmcp==4.0.5

print("Listo. Si no viste ningun error rojo, seguimos.")


In [ ]:
# ===========================================================================
# Imports y claves (PARA CASA: es puro setup, no hay nada que aprender aca)
# ===========================================================================
import base64
import json
import os
import warnings

import requests
from dotenv import load_dotenv
from IPython.display import Image, display
from openai import OpenAI

# Estas dos librerias avisan de cosas que no podemos arreglar desde nuestro codigo
# (una de ellas llama internamente a una funcion vieja), asi que las silenciamos
# para que no nos ensucien la pantalla. Solo estas dos, nunca todos los warnings:
# si algo se rompe de verdad, queremos verlo.
from langgraph.warnings import LangGraphDeprecatedSinceV10

warnings.filterwarnings("ignore", category=LangGraphDeprecatedSinceV10)
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

# ===========================================================================
# Las claves salen del .env que reparte Axel (NARRAR)
# ===========================================================================
# Nunca escribas una clave en el notebook. Si la escribis, queda en el archivo,
# y el archivo lo compartis. El .env se queda en tu maquina y no se sube nunca.
# find_dotenv busca el .env en la carpeta de trabajo. En Colab, si subiste el
# archivo con el panel de la izquierda, queda en /content y lo encuentra solo.
# Si te dice "Missing credentials", el .env no esta donde el notebook lo busca.
load_dotenv()

MODEL = "gpt-4o-mini"          # barato y suficiente para todo el taller
client = OpenAI()              # toma OPENAI_API_KEY del entorno, sin pasarsela a mano
SERPER_KEY = os.environ["SERPER_API_KEY"]   # esta si la usamos explicitamente

# ===========================================================================
# Helper para ver los diagramas (PARA CASA)
# ===========================================================================
# Colab no dibuja Mermaid en las celdas de texto, asi que traemos el diagrama
# del repo y lo renderizamos como imagen.
DIAGRAMAS = (
    "https://raw.githubusercontent.com/"
    "axel-sirota/nerdearla_concierge_agent/main/diagrams"
)


def mostrar_diagrama(slug: str) -> None:
    """Trae un diagrama del repo y lo dibuja como imagen."""
    fuente = requests.get(f"{DIAGRAMAS}/{slug}.mmd", timeout=30)
    if fuente.status_code != 200:
        # Si el diagrama todavia no esta en el repo, avisamos en texto en lugar
        # de dibujar la pagina de error de GitHub.
        print(f"(El diagrama '{slug}' todavia no esta publicado.)")
        return
    codificado = base64.urlsafe_b64encode(fuente.text.encode()).decode()
    imagen = requests.get(f"https://mermaid.ink/img/{codificado}", timeout=30)
    display(Image(data=imagen.content))


print(f"Todo listo. Modelo: {MODEL}")


## Seccion 1: El caso

Tres pedidos distintos en una sola frase. Miremos el problema antes de escribir codigo.

<!-- /build-notebook llena esta seccion desde su plan. -->


## Seccion 2: Un agente es un loop

Python puro. Sin framework, sin nada importado que esconda el truco.

<!-- /build-notebook llena esta seccion desde su plan. -->


Antes de tocar un framework, quiero que veamos que hay abajo. Porque lo que sigue es la
parte que despues todos los frameworks te esconden, y si no la viste una vez a mano, el
framework te va a parecer magia.

Arranquemos por el problema. Le vamos a preguntar a `gpt-4o-mini` cuanto sale un vuelo a
Bariloche. Nada mas: el modelo solo, sin tools, sin nada.

Fijate en dos cosas cuando corra la celda que viene:

1. Te contesta con un numero.
2. Ese numero es inventado.

No esta mintiendo a proposito. El modelo no tiene forma de ir a mirar. Lo unico que sabe
hacer es predecir texto plausible, y un precio plausible es exactamente eso: plausible.


In [ ]:
# ===========================================================================
# EL PROBLEMA: el modelo solo, sin herramientas (NARRAR)
# ===========================================================================
# Ojo: no le pasamos ninguna tool. Es el modelo pelado, como lo usaste siempre.
# Le preguntamos un precio concreto, de esta semana, del mundo real.

respuesta = client.chat.completions.create(
    model=MODEL,
    messages=[
        # El system prompt define el personaje. Nada mas que eso por ahora.
        {"role": "system", "content": "Sos un agente de viajes argentino. Respondes corto."},
        {"role": "user", "content": "Cuanto sale un vuelo de Buenos Aires a Bariloche la "
                                    "semana que viene? Dame precio y aerolinea."},
    ],
    # PARA CASA: fijate que no hay parametro `tools` en esta llamada. Esa ausencia es
    # todo el punto de la celda: el modelo no tiene ninguna via para salir a buscar.
)

print(respuesta.choices[0].message.content)


### Lo que dijo el modelo, y lo que cuesta de verdad

En una corrida de prueba el modelo tiro un rango de ARS 15.000 a 30.000. Los precios
reales, buscados ese mismo minuto: **ARS 50.550, 57.186 y 70.457**. Se equivoco por dos a
cinco veces, y lo dijo con total seguridad.

Y aca esta la parte importante, porque es facil sacar la conclusion equivocada: el problema
no es que el numero este mal. El problema es que **el modelo no tiene forma de ir a
buscarlo**. Si le pegas el precio en el prompt arreglas esta pregunta y ninguna otra: no
sabes de antemano si te va a preguntar por vuelos, por hoteles o por que hacer el finde.

Lo que necesitamos es darle una forma de pedirnos que busquemos nosotros. Eso es un agente,
y el loop completo entra en un diagrama.

<!-- DIAGRAM: sequenceDiagram del loop del agente. Cuatro participantes: Vos, Tu codigo, Modelo, serper.dev. El usuario pide vuelos a Tu codigo; Tu codigo manda mensajes mas la lista de tools al Modelo; el Modelo devuelve tool_calls pidiendo buscar_vuelos; una Note sobre Tu codigo aclara que el modelo NO ejecuta nada; Tu codigo llama a serper.dev; serper devuelve resultados; Tu codigo manda el resultado al Modelo como mensaje role tool con su tool_call_id; el Modelo devuelve la respuesta final en texto; Tu codigo se la pasa al usuario. Es el diagrama mas importante del taller. -->


In [ ]:
# El diagrama mas importante del taller. Miralo dos veces.
mostrar_diagrama("agente-loop")


El diagrama muestra el viaje completo de una pregunta. Quedate con el paso del medio:
cuando el modelo contesta `tool_calls`, **no ejecuto nada**. Te devolvio un pedido escrito:
"llama a `buscar_vuelos` con estos argumentos". El que ejecuta sos vos. Siempre.


In [ ]:
# ===========================================================================
# LAYER 0: el wrapper de busqueda. Esta parte es solo algoritmica (NARRAR)
# ===========================================================================
SERPER_URL = "https://google.serper.dev/search"


def _serper(query: str, n: int = 5) -> list[dict]:
    """Consulta serper.dev y devuelve los resultados organicos crudos."""
    # PARA CASA: serper.dev es Google servido como JSON. Le mandamos un POST y nos
    # devuelve la misma pagina de resultados que verias en el navegador, ya parseada.
    respuesta = requests.post(
        SERPER_URL,
        # La clave va en un header, no en la URL: asi no queda en logs ni en el historial.
        headers={"X-API-KEY": SERPER_KEY,
                 "Content-Type": "application/json"},
        # gl="ar" y hl="es" son lo que hace que esto sirva para Argentina: resultados
        # geolocalizados y en español. Sin eso te llegan precios en dolares y paginas
        # en ingles.
        json={"q": query, "gl": "ar", "hl": "es", "num": n},
        # Timeout siempre. Un agente que se cuelga esperando una API es un agente colgado.
        timeout=20,
    )
    # Si la clave esta vencida o mal, esto explota ACA con un mensaje claro (403), en vez
    # de devolver una lista vacia y hacerte creer que Bariloche no existe.
    respuesta.raise_for_status()
    # "organic" son los resultados de siempre, los diez azules. El .get con [] por
    # defecto nos cubre de una busqueda sin resultados.
    return respuesta.json().get("organic", [])


def _normalizar(resultados: list[dict]) -> list[dict]:
    """Deja solo titulo, resumen y link de cada resultado."""
    # PARA CASA: el JSON de Google trae un monton de campos que no nos importan.
    # Nos quedamos con tres. Esto NO es para ahorrar tokens (medido: apenas un 6% menos):
    # es para que el modelo, y vos, lean algo prolijo en vez de un bloque de ruido.
    return [
        {
            "titulo": r["title"],
            # "snippet" casi siempre viene, pero no esta garantizado. El .get con ""
            # evita un KeyError en vivo por un resultado raro.
            "resumen": r.get("snippet", ""),
            "link": r["link"],
        }
        for r in resultados
    ]


print("Wrapper listo.")


In [ ]:
# ===========================================================================
# LAYER 1: las dos tools de verdad (NARRAR)
# ===========================================================================
# Estas dos funciones son las tools. No tienen nada especial: son funciones de Python
# comunes. Lo unico que las hace "tools" es que despues se las vamos a describir al
# modelo.


def buscar_vuelos(origen: str, destino: str) -> list[dict]:
    """Busca vuelos entre dos ciudades argentinas.

    Args:
        origen: ciudad de salida, por ejemplo "Buenos Aires"
        destino: ciudad de llegada, por ejemplo "Bariloche"
    """
    # PARA CASA: el docstring no es decoracion. En un rato se lo vamos a copiar tal cual
    # al schema, y el modelo lo lee para decidir cual tool usar. Docstring vago, tool mal
    # elegida. Escribilos como si el lector fuera el modelo, porque lo es.
    return _normalizar(_serper(f"vuelos {origen} a {destino} precio"))


def buscar_hoteles(ciudad: str, zona: str = "centro") -> list[dict]:
    """Busca hoteles en una ciudad argentina.

    Args:
        ciudad: por ejemplo "Bariloche"
        zona: barrio o area, por defecto "centro"
    """
    # PARA CASA: `zona` tiene default. Si el modelo no lo manda, Python pone "centro".
    # Es un default de Python comun, el modelo no se entera ni le importa.
    return _normalizar(_serper(f"hoteles {zona} {ciudad} precio"))


# Probemos una, a mano, sin agente ni modelo de por medio. Esto es codigo y nada mas.
for vuelo in buscar_vuelos("Buenos Aires", "Bariloche")[:3]:
    print("-", vuelo["titulo"])
    print("  ", vuelo["resumen"][:90])


In [ ]:
# ===========================================================================
# EL SCHEMA A MANO: esto es lo que el modelo realmente ve (NARRAR)
# ===========================================================================
# Aca esta el truco entero, y lo escribimos a mano a proposito, una sola vez en todo
# el taller. Un framework te genera esto solo desde el docstring y los type hints,
# y esta bien que lo haga. Pero si nunca lo viste, no sabes que existe.
#
# El modelo NO recibe tu funcion de Python. No puede. Vive en otra computadora.
# Lo unico que recibe es esta descripcion en JSON: como se llama, para que sirve,
# y que argumentos acepta.

ESQUEMA_BUSCAR_VUELOS = {
    # "function" es el unico tipo que vamos a usar hoy.
    "type": "function",
    "function": {
        # El nombre TIENE que coincidir exactamente con la clave del dict de tools que
        # le pasemos al agente. Si no coincide, el dispatch no encuentra la funcion.
        "name": "buscar_vuelos",
        # Esta linea es la que el modelo lee para elegir. Es el docstring, copiado.
        # Es el campo mas importante del schema y el que mas gente escribe al apuro.
        "description": "Busca vuelos entre dos ciudades argentinas.",
        # Los argumentos, en JSON Schema.
        "parameters": {
            "type": "object",
            "properties": {
                "origen": {"type": "string",
                           "description": "ciudad de salida, por ejemplo Buenos Aires"},
                "destino": {"type": "string",
                            "description": "ciudad de llegada, por ejemplo Bariloche"},
            },
            # Lo que el modelo esta obligado a mandar. Lo que no este aca es opcional.
            "required": ["origen", "destino"],
        },
    },
}

# Miralo impreso. Son 24 lineas de JSON. Eso es toda la "magia" de las tools.
print(json.dumps(ESQUEMA_BUSCAR_VUELOS, indent=2, ensure_ascii=False))


In [ ]:
# ===========================================================================
# LA CLASE, PARTE 1: que se guarda un agente (NARRAR)
# ===========================================================================
class AgenteVuelos:
    """Un agente es esto: un loop que le pregunta al modelo que tool usar."""

    def __init__(self, tools: dict[str, callable], system: str, model: str = MODEL):
        # `tools` es un diccionario de nombre a funcion de Python:
        #     {"buscar_vuelos": buscar_vuelos}
        # Esta es la tabla de dispatch. Cuando el modelo pida "buscar_vuelos",
        # buscamos esa clave aca y llamamos a la funcion. No hay nada mas que eso.
        self.tools = tools
        self.system = system
        self.model = model
        # PARA CASA: la conversacion es una lista de Python comun. No es un objeto
        # con estado oculto ni un "contexto" magico. Es una lista, y la vamos a hacer
        # crecer nosotros a mano. Podes imprimirla cuando quieras: lo vas a hacer en
        # un ejercicio al final de esta seccion.
        self.messages: list[dict] = [{"role": "system", "content": system}]

    def _tool_schemas(self) -> list[dict]:
        """Traduce las funciones de Python al formato que espera OpenAI."""
        # Devolvemos la lista de schemas que armamos a mano en la celda anterior.
        # En un framework esta funcion no existe: se genera sola. Aca la vemos.
        return [ESQUEMA_BUSCAR_VUELOS]


print("Parte 1 lista: ya sabe que guardar. Todavia no sabe hacer nada.")


In [ ]:
# ===========================================================================
# LA CLASE, PARTE 2: el loop. Esto es todo lo que un agente es (NARRAR)
# ===========================================================================
# Truco de notebook: reabrimos la clase heredando de si misma, solo para poder mostrar
# el loop en su propia celda. En un archivo .py los dos bloques serian una sola clase.
class AgenteVuelos(AgenteVuelos):

    def run(self, pregunta: str, max_turns: int = 5) -> str:
        """El loop. Esto es todo lo que un agente es."""
        # La pregunta del usuario entra a la conversacion como un mensaje mas.
        self.messages.append({"role": "user", "content": pregunta})

        turn = 0
        # `max_turns` es el freno de mano. Sin un techo duro no tenes una garantia,
        # tenes una esperanza: si el modelo se obstina en pedir tools, el loop no para.
        # En produccion esto se llama "bounded execution" y es obligatorio.
        while turn < max_turns:
            turn += 1

            # --- PASO 1: le preguntamos al modelo, con la lista de tools adjunta -----
            resp = client.chat.completions.create(
                model=self.model,
                messages=self.messages,          # TODA la conversacion, cada vez
                tools=self._tool_schemas(),      # el JSON de la celda anterior
            )
            msg = resp.choices[0].message

            # --- PASO 2: el modelo pidio una tool, o ya contesto? -------------------
            if not msg.tool_calls:
                # No pidio nada mas: lo que hay en `content` es la respuesta final.
                # Aca sale el loop. Una pregunta con una tool tarda dos vueltas:
                # en la primera pide la tool, en la segunda escribe la respuesta.
                self.messages.append({"role": "assistant", "content": msg.content})
                return msg.content

            # --- PASO 3: appendear el mensaje del modelo ANTES que nada --------------
            # OBLIGATORIO y facil de olvidar. Si mandas la respuesta de la tool sin haber
            # appendeado primero este mensaje, la API te devuelve 400:
            #   "messages with role 'tool' must be a response to a preceeding message
            #    with 'tool_calls'"
            # `exclude_none=True` saca los campos vacios (como content, que viene en None
            # cuando el modelo pide tools) que la API no quiere recibir.
            self.messages.append(msg.model_dump(exclude_none=True))

            # --- PASO 4: NOSOTROS ejecutamos. El modelo nunca ejecuto nada -----------
            # El `for` no es decoracion: el modelo puede pedir VARIAS tools en un solo
            # mensaje, y lo hace seguido. Y hay que contestarle a CADA tool_call_id, si
            # te salteas uno la API devuelve 400:
            #   "must be followed by tool messages responding to each 'tool_call_id'"
            for call in msg.tool_calls:
                nombre = call.function.name
                # Los argumentos vienen como STRING con JSON adentro, no como dict.
                args = json.loads(call.function.arguments)
                print(f"   [turno {turn}] el modelo pidio: {nombre}({args})")

                # La tabla de dispatch en accion. Una busqueda en un dict y una llamada.
                # Esta linea es el corazon del agente, y es una linea de Python comun.
                resultado = self.tools[nombre](**args)

                # --- PASO 5: devolvemos el resultado como un mensaje mas -------------
                # `content` tiene que ser un STRING. Si le pasas la lista de dicts tal
                # cual, la API te devuelve 400 con un error medio cifrado:
                #   "Missing required parameter: 'messages[N].content[0].type'"
                # ensure_ascii=False para que los acentos se lean, aca y en pantalla.
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,   # ata esta respuesta a ESE pedido
                    "content": json.dumps(resultado, ensure_ascii=False),
                })
            # Volvemos al while: el modelo ahora ve el resultado y decide otra vez.

        # Si llegamos aca, se agotaron los turnos sin respuesta final. Devolvemos un
        # string igual, porque run() promete devolver un string siempre.
        return "Me quede sin turnos."


print("Parte 2 lista: ahora si es un agente.")


In [ ]:
# ===========================================================================
# A CORRERLO (NARRAR)
# ===========================================================================
# Le damos UNA sola tool, aunque `buscar_hoteles` ya existe y esta ahi al lado.
# Eso es a proposito, y en un rato vamos a ver por que.
agente_vuelos = AgenteVuelos(
    tools={"buscar_vuelos": buscar_vuelos},
    system=("Sos un agente de viajes argentino. Usa las tools que tengas para buscar "
            "datos reales antes de contestar. Respondes corto y en español rioplatense."),
)

# La pregunta de siempre, la de Bariloche.
respuesta_agente = agente_vuelos.run("Buscame vuelos de Buenos Aires a Bariloche.")

print()
print(respuesta_agente)
print()
# PARA CASA: cinco mensajes, y cada uno es un paso del diagrama:
# system (el personaje), user (la pregunta), assistant (pidio la tool),
# tool (lo que encontramos), assistant (la respuesta final).
print("Mensajes en la conversacion:", len(agente_vuelos.messages))
print("Roles:", [m["role"] for m in agente_vuelos.messages])


### Lo que acabamos de construir

Contemos las lineas que importan: un `while`, una llamada a la API, un `if` que pregunta si
hay `tool_calls`, un `for` que despacha, y una lista a la que le hacemos `append`. Eso es
un agente completo y funcionando.

Y ahora mira los imports de esta seccion: `json`, `os`, `requests`, `openai`. **No hay ni
un framework.** Ninguna de las cinco lineas de arriba nos la dio una libreria de agentes.

<!-- DIAGRAM: graph LR con la anatomia de la clase AgenteVuelos. Un subgraph AgenteVuelos con el while turn menor a max_turns en el centro, que lleva a chat.completions.create, que lleva a un rombo de decision hay tool_calls; la rama no va a return contenido; la rama si va a dispatch self.tools, de ahi a append role tool, y de ahi vuelve al while cerrando el ciclo. Afuera del subgraph, dos cajas punteadas: OpenAI conectada a chat.completions.create, y serper.dev conectada al dispatch. Conecta el codigo que acaban de ver con el Diagrama 2. -->


In [ ]:
# El mismo loop del diagrama anterior, pero dibujado como codigo.
mostrar_diagrama("agente-anatomia")


El diagrama toma el mismo loop del Diagrama 2 y lo dibuja como codigo: donde esta el
`while`, donde el `if`, y que dos cosas viven **afuera** de tu proceso. Las dos cajas
punteadas son las unicas piezas que no controlas: la API de OpenAI y serper.dev.

Tres cosas para llevarse de esta seccion:

1. **El modelo nunca ejecuta nada.** Te devuelve un pedido. El que ejecuta sos vos, en el
   `for`, con un dict y una llamada a funcion.
2. **La conversacion es una lista.** Le hacemos `append` a mano, en un orden que la API
   exige: primero el mensaje del modelo, despues una respuesta por cada `tool_call_id`.
3. **El loop termina** porque el modelo deja de pedir tools, o porque se acaban los turnos.
   Nunca dejes un agente sin techo.

Todo esto que acabamos de escribir a mano, un framework ya lo tiene resuelto. Vamos a ver
cuanto codigo nos borra, y sobre todo: ahora sabes exactamente que codigo te esta borrando.


In [ ]:
# ===========================================================================
# MINI EJERCICIOS (opcionales, para despues)
# ===========================================================================
# Ninguno de estos es necesario para seguir el taller. El notebook corre igual si los
# salteas. Las soluciones estan todas al final del notebook.

# EJERCICIO 1 (opcional, para despues)
# Agregale `buscar_hoteles` al agente, que ya esta definida y sin usar.
# Son dos cambios: un schema nuevo en `_tool_schemas` (copia el de vuelos y cambiale
# nombre, description y parameters), y una clave mas en el dict `tools`.
# Despues pedile: "vuelos de Buenos Aires a Bariloche y un hotel cerca del centro".
# Pista: fijate si el modelo pide las dos tools en el mismo turno o en turnos separados.
# Solucion al final del notebook.

# EJERCICIO 2 (opcional, para despues)
# Imprimi `agente_vuelos.messages` y leete la conversacion completa, mensaje por mensaje.
# Es una lista de Python comun, no hay nada escondido ahi adentro.
# Pista: para que se lea, imprimi el rol y los primeros 60 caracteres de cada mensaje.
# Ojo con el mensaje del assistant: despues de model_dump es un dict, asi que las
# tool_calls se leen con corchetes, m["tool_calls"], no con punto.
# Solucion al final del notebook.

# EJERCICIO 3 (opcional, para despues)
# Corre el agente con `max_turns=1` y mira que pasa.
# Pista: el modelo pide la tool en el turno 1, asi que nunca llega a escribir la respuesta
# final. Fijate que devuelve `run()` y como quedo la lista de mensajes, cortada al medio.
# Esto es exactamente para lo que existe el freno de mano.
# Solucion al final del notebook.


## Seccion 3: Lo mismo, pero en LangGraph

El mismo agente, migrado. Ahora si podes leer el framework, porque ya lo hiciste a mano.

<!-- /build-notebook llena esta seccion desde su plan. -->


Recien escribimos un agente sin framework. Funciona. Y funciona porque escribimos, a mano,
todo esto:

- el `while` con el contador de turnos
- el `if not msg.tool_calls` para saber cuando parar
- el `json.loads` de los argumentos que manda el modelo
- el diccionario para despachar la funcion por nombre
- y **24 lineas de JSON Schema** para describir UNA sola tool

Ese JSON es el que mas duele. Fijate que la descripcion que pusimos a mano ("ciudad de
salida, por ejemplo Buenos Aires") ya estaba escrita en el docstring de `buscar_vuelos`.
La escribimos dos veces: una para los humanos y una para el modelo.

Ahora hacemos el mismo agente en LangGraph. Mismo `buscar_vuelos`, misma pregunta, mismos
datos de serper. La gracia no es que quede mas corto. La gracia es que **ahora podes leer
el framework**, porque hace diez minutos escribiste lo que hay adentro.


In [ ]:
# ===========================================================================
# EL SCHEMA A MANO CONTRA EL SCHEMA GENERADO (NARRAR)
# ===========================================================================
# Arrancamos por lo que mas trabajo nos dio: el JSON Schema.
# LangChain trae una funcion que convierte una funcion de Python en una tool.
# Le pasamos parse_docstring=True a proposito: sin eso, LangChain manda el docstring
# entero como una sola descripcion y PIERDE la descripcion de cada argumento.
# Con eso puesto, lee el bloque "Args:" y arma el mismo schema que tipeamos a mano.
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

# OJO: no redefinimos buscar_vuelos. Es la misma funcion de la seccion anterior, envuelta.
vuelos_tool = tool(buscar_vuelos, parse_docstring=True)

# El schema que tipeamos a mano, sacado del agente que ya corrimos.
a_mano = AgenteVuelos({"buscar_vuelos": buscar_vuelos}, "comparacion")._tool_schemas()[0]

# El schema que LangChain genero leyendo el docstring en castellano.
generado = convert_to_openai_tool(vuelos_tool)

print("=== LO QUE ESCRIBIMOS A MANO ===")
print(json.dumps(a_mano, indent=2, ensure_ascii=False))
print()
print("=== LO QUE GENERO LANGCHAIN DEL DOCSTRING ===")
print(json.dumps(generado, indent=2, ensure_ascii=False))
print()

# PARA CASA: no pedimos que sean identicos caracter por caracter. Los textos salen del
# docstring, asi que son mas completos. Lo que importa es que el modelo recibe la misma
# forma: mismo nombre, mismos argumentos, mismos obligatorios.
print("mismo nombre de tool:",
      a_mano["function"]["name"] == generado["function"]["name"])
print("mismos argumentos:",
      set(a_mano["function"]["parameters"]["properties"])
      == set(generado["function"]["parameters"]["properties"]))
print("mismos obligatorios:",
      a_mano["function"]["parameters"]["required"]
      == generado["function"]["parameters"]["required"])


### El loop que escribimos, y el mismo loop en LangGraph

Las dos mitades van juntas en pantalla. A la izquierda lo de la seccion anterior, a la
derecha lo que vamos a construir ahora. Las cajas se corresponden una a una.

<!-- DIAGRAM: 4 - side-by-side. A la izquierda el loop a mano del Beat 1 con las etiquetas de su propio codigo (pregunta, client.chat.completions.create, if not msg.tool_calls return, for call in msg.tool_calls, self.tools[nombre](**args), while turn < max_turns). A la derecha el grafo que LangGraph compila de verdad, extraido con get_graph(): nodos __start__, model, tools, __end__; aristas __start__ -> model directa, model -> tools condicional, model -> __end__ condicional, tools -> model el loop. Las cajas de un lado se corresponden una a una con las del otro. Las dos mitades TIENEN que estar en pantalla juntas. -->


In [ ]:
# Las dos mitades juntas: a la izquierda lo nuestro, a la derecha el grafo.
mostrar_diagrama("langgraph")


Los nodos del grafo no son invento nuestro: se llaman `model` y `tools`, y los vas a ver
impresos con esos nombres cuando corramos el agente en un rato.


In [ ]:
# ===========================================================================
# EL AGENTE, EN CINCO LINEAS (NARRAR)
# ===========================================================================
# create_agent es la forma actual de armar un agente en LangChain 1.x.
# Si viste tutoriales con create_react_agent de langgraph.prebuilt: esa quedo
# deprecada, te tira un warning en pantalla y apunta aca. Usamos la actual.
from langchain.agents import create_agent

# El mismo prompt para los dos agentes, para que la comparacion sea honesta.
SYSTEM_VUELOS = (
    "Sos un asistente de viajes argentino. Contesta corto y en español rioplatense."
)

# Y esto es TODO el agente. Lo que antes fueron 28 lineas de clase.
agente_vuelos_lg = create_agent(
    # "openai:" + el modelo. Reusamos la constante MODEL para que el notebook
    # tenga un solo lugar donde cambiar de modelo.
    model=f"openai:{MODEL}",
    # La lista de tools. Agregar otra tool es agregar un elemento a esta lista:
    # nada de JSON a mano. Acordate de esto en la seccion que viene.
    tools=[vuelos_tool],
    # Lo que antes era self.messages[0]. Ojo: en LangGraph el system prompt
    # NO vive dentro de la lista de mensajes, va por separado.
    system_prompt=SYSTEM_VUELOS,
)

# No es un servicio ni un runtime raro: es un objeto de Python con .invoke().
print("tipo:", type(agente_vuelos_lg).__name__)
print("nodos del grafo:", list(agente_vuelos_lg.nodes))


In [ ]:
# ===========================================================================
# EL LOOP, POR DENTRO (NARRAR)
# ===========================================================================
# Hasta aca te pedimos que creas que adentro hay un loop. Ahora lo miramos.
# .stream() nos entrega el estado despues de cada nodo, asi que podemos imprimir
# el recorrido: es el diagrama del loop, narrado por el framework mismo.
PREGUNTA_VUELOS = "Buscame vuelos de Buenos Aires a Bariloche"

for paso in agente_vuelos_lg.stream(
    {"messages": [{"role": "user", "content": PREGUNTA_VUELOS}]}
):
    # paso es un dict con UNA clave: el nombre del nodo que acaba de correr.
    for nodo, estado_nodo in paso.items():
        ultimo = estado_nodo["messages"][-1]
        if getattr(ultimo, "tool_calls", None):
            # El nodo model decidio que necesita una tool. Esto es exactamente
            # el msg.tool_calls que leiamos a mano antes.
            pedido = ultimo.tool_calls[0]
            print(f"  [{nodo}] el modelo pide: {pedido['name']}({pedido['args']})")
        elif type(ultimo).__name__ == "ToolMessage":
            # El nodo tools ejecuto la funcion y metio el resultado como mensaje.
            # Esto es nuestro self.messages.append({"role": "tool", ...}).
            print(f"  [{nodo}] la tool devolvio {len(ultimo.content)} caracteres")
        else:
            # El modelo contesto sin pedir nada mas: el loop termina.
            # Esto es nuestro if not msg.tool_calls: return msg.content.
            print(f"  [{nodo}] el modelo contesta, sin pedir mas tools")


### Caja por caja

| Lo que escribimos a mano | Como se llama en LangGraph |
|---|---|
| `client.chat.completions.create(...)` | el nodo `model` |
| `for call in msg.tool_calls:` mas `self.tools[nombre](**args)` | el nodo `tools` |
| `if not msg.tool_calls: return msg.content` | la arista condicional `model -> __end__` |
| el `while` que vuelve a preguntar | la arista `tools -> model` |
| `self.messages` que iba creciendo | el estado del grafo, la clave `messages` |
| las 24 lineas de JSON Schema | el docstring, parseado por `tool(...)` |
| `max_turns=5` | `recursion_limit`, o `ModelCallLimitMiddleware(run_limit=5)` |
| `self.messages[0]` con el system | el argumento `system_prompt`, que va aparte |

No hay ninguna fila sin par. Eso es lo que compramos: no un concepto nuevo, el mismo
concepto con el plomero ya escrito.


In [ ]:
# ===========================================================================
# LOS DOS AGENTES, LA MISMA PREGUNTA (NARRAR)
# ===========================================================================
# La prueba honesta: la misma pregunta a los dos agentes, uno al lado del otro.
# Los dos usan la misma funcion buscar_vuelos y le pegan al mismo serper.

# El de la seccion anterior: devuelve directamente el string final.
respuesta_a_mano = agente_vuelos.run(PREGUNTA_VUELOS)

# El de LangGraph: devuelve el estado completo, con TODOS los mensajes.
# La respuesta final es el contenido del ultimo mensaje.
estado = agente_vuelos_lg.invoke(
    {"messages": [{"role": "user", "content": PREGUNTA_VUELOS}]}
)
respuesta_langgraph = estado["messages"][-1].content

print("=== A MANO ===")
print(respuesta_a_mano[:400])
print()
print("=== LANGGRAPH ===")
print(respuesta_langgraph[:400])
print()

# PARA CASA: si te interesa ver el viaje completo, esta todo en el estado.
# Fijate que el system prompt NO aparece: LangGraph lo guarda aparte.
print("mensajes que quedaron en el estado:",
      [type(m).__name__ for m in estado["messages"]])


### Que compramos y que nos esconde

Compramos el plomero: 28 lineas de clase se volvieron 5, el JSON Schema sale del
docstring, y el estado de la conversacion lo maneja el grafo.

Tambien compramos cosas que no vamos a usar hoy y que valen para cuando esto pase de demo
a algo real: guardar la conversacion entre llamadas con un `checkpointer`, cortar el agente
con `ModelCallLimitMiddleware`, resumir el historial cuando se hace largo, o pedir
aprobacion humana antes de ejecutar una tool sensible.

Y nos esconde el loop. Por eso lo escribimos primero. Cuando un agente en LangGraph se te
va de las manos, lo que tenes que mirar es lo mismo que mirabamos a mano: que tool pidio el
modelo, con que argumentos, y que le devolvio la tool.

Hay una cosa mas, y es la que nos mete en problemas en la proxima seccion. Agregar una tool
ahora sale **una linea**: un elemento mas en `tools=[...]`. Cuando algo sale tan barato,
uno agrega. Y despues agrega otra. Vamos a ver que pasa cuando un solo agente termina con
seis.


In [ ]:
# ===========================================================================
# MINI EJERCICIOS (opcionales, para despues)
# ===========================================================================
# Las soluciones estan todas al final del notebook.

# EJERCICIO 4 (opcional, para despues)
# Arma otro agente igual a agente_vuelos_lg pero cambiandole el system_prompt
# por "Contesta en una sola linea, sin listas". Hacele la misma pregunta y compara.
# Es el unico parametro que tocaste y la respuesta cambia entera: el prompt es
# parte del agente, no un adorno.
# Solucion al final del notebook.

# EJERCICIO 5 (opcional, para despues)
# Imprimi agente_vuelos_lg.get_graph().draw_mermaid() y busca en el texto que sale
# la linea "tools --> model". Esa flecha es el while de la seccion anterior.
# No necesitas instalar nada para esto.
# Solucion al final del notebook.

# EJERCICIO 6 (opcional, para despues, y el mas interesante de los tres)
# Envolve buscar_vuelos SIN parse_docstring=True, convertilo con convert_to_openai_tool
# y compara el schema con el de la celda de arriba. Vas a ver que las descripciones de
# cada argumento desaparecen y que el docstring entero queda apretado en un solo campo.
# Moraleja: el framework no adivina nada, te parsea el docstring, y le podes pedir que
# lo parsee mal.
# Solucion al final del notebook.


## Seccion 4: Un agente con demasiadas tools

Le damos seis tools a un solo agente y lo vemos elegir mal, en vivo.

<!-- /build-notebook llena esta seccion desde su plan. -->


Hasta aca tenemos dos tools que andan: `buscar_vuelos` y `buscar_hoteles`. Y tenemos el
loop, que lo escribimos nosotros, asi que sabemos exactamente que hace.

Ahora viene el pedido completo:

> "Buscame vuelos a Bariloche, un hotel cerca del centro y que hacer el finde."

Que harias vos? Lo mismo que hacemos todos la primera vez: un solo agente, y le colgamos
todas las tools que hacen falta. Es una linea de codigo. Vamos a hacerlo, y vamos a ver
por que es una trampa.


### Como llegamos a seis tools

Nadie arranca con seis tools. Se llega. Fijate la secuencia, que la vas a reconocer:

1. Arrancas con `buscar_vuelos`. Anda.
2. Un usuario pide algo mas barato. Alguien agrega `buscar_vuelos_baratos`. Nadie borra
   la primera, porque hay codigo que la usa.
3. Lo mismo del otro lado: `buscar_hoteles`, y despues `buscar_alojamiento`.
4. Aparece `buscar_transporte`, que nadie sabe bien si son vuelos o micros.
5. Y `buscar_actividades`, que es la unica que no se pisa con nada.

Seis tools. Dos pares que hacen lo mismo. Y el agente sigue siendo el mismo agente.


In [ ]:
# ===========================================================================
# LAS SEIS TOOLS (NARRAR)
# ===========================================================================
# Dos ya las tenemos de antes: buscar_vuelos y buscar_hoteles.
# Aca definimos las otras cuatro, y todas usan el mismo wrapper _serper de siempre.
#
# Prestá atencion a los docstrings. El modelo elige la tool LEYENDO ESTO.
# Para el modelo, el docstring no es documentacion: es la interfaz.


def buscar_vuelos_baratos(origen: str, destino: str) -> list[dict]:
    """Busca vuelos entre dos ciudades argentinas, con los mejores precios.

    Args:
        origen: ciudad de salida, por ejemplo "Buenos Aires"
        destino: ciudad de llegada, por ejemplo "Bariloche"
    """
    # Se pisa con buscar_vuelos A PROPOSITO. Y fijate que el docstring agrega
    # "con los mejores precios": esa frase de mas es toda la diferencia, y es la
    # que despues nos va a costar caro.
    return _normalizar(_serper(f"vuelos baratos {origen} {destino}"))


def buscar_alojamiento(ciudad: str, zona: str = "centro") -> list[dict]:
    """Busca alojamiento en una ciudad argentina, con los mejores precios.

    Args:
        ciudad: por ejemplo "Bariloche"
        zona: barrio o area, por defecto "centro"
    """
    # El gemelo de buscar_hoteles. Mismo truco: "con los mejores precios".
    return _normalizar(_serper(f"alojamiento barato {zona} {ciudad}"))


def buscar_transporte(origen: str, destino: str) -> list[dict]:
    """Busca transporte entre dos ciudades argentinas.

    Args:
        origen: ciudad de salida, por ejemplo "Buenos Aires"
        destino: ciudad de llegada, por ejemplo "Bariloche"
    """
    # Esta es la vaga: "transporte" puede ser un vuelo, un micro o un remis.
    # En un sistema real es la tool que nadie se anima a borrar.
    return _normalizar(_serper(f"como llegar {origen} a {destino}"))


def buscar_actividades(ciudad: str) -> list[dict]:
    """Busca actividades y paseos en una ciudad argentina.

    Args:
        ciudad: por ejemplo "Bariloche"
    """
    # La unica de las seis que no se pisa con ninguna otra. Acordate de esta,
    # porque al final va a ser la unica que el agente elige bien.
    return _normalizar(_serper(f"que hacer en {ciudad} fin de semana"))


# El catalogo completo: las dos de siempre mas las cuatro nuevas.
# Las claves son los nombres que ve el modelo; los valores, las funciones que
# nosotros vamos a ejecutar cuando el modelo las pida.
TOOLS_MUCHAS = {
    "buscar_vuelos": buscar_vuelos,                    # la original
    "buscar_vuelos_baratos": buscar_vuelos_baratos,    # gemela de la anterior
    "buscar_hoteles": buscar_hoteles,                  # la original
    "buscar_alojamiento": buscar_alojamiento,          # gemela de la anterior
    "buscar_transporte": buscar_transporte,            # vaga
    "buscar_actividades": buscar_actividades,          # la unica clara
}

print(f"El agente va a tener {len(TOOLS_MUCHAS)} tools:")
for nombre in TOOLS_MUCHAS:
    print(f"  - {nombre}")


### El catalogo, dibujado

<!-- DIAGRAM: un agente en el centro con las 6 tools colgando. buscar_vuelos y buscar_vuelos_baratos resaltadas como par que se pisa; buscar_hoteles y buscar_alojamiento resaltadas como el otro par. buscar_transporte marcada como vaga. buscar_actividades la unica limpia. -->


In [ ]:
# Mira las dos parejas resaltadas antes de seguir.
mostrar_diagrama("muchas-tools-catalogo")


Un agente, seis tools. Los dos pares resaltados son los que hacen lo mismo con dos nombres
distintos.

### 🎲 Antes de correrlo: apostemos

Pará aca un segundo. No corras la celda todavia.

El pedido va a ser el de siempre: vuelos a Bariloche, hotel en el centro, y que hacer el
finde. El agente tiene estas seis tools y un system prompt que le pide priorizar el precio,
que es lo que le pediria cualquier producto real.

**Que pensas que va a pasar?**

1. Elige bien las tres tools que corresponden.
2. Se cuelga y no llama ninguna.
3. Contesta cualquier cosa, un delirio.
4. Contesta perfecto, y usa las tools equivocadas.

Quedate con tu numero. Ahora si, corremos.


In [ ]:
# ===========================================================================
# LA CLASE: es LA MISMA de antes, heredada (NARRAR)
# ===========================================================================
# Aca esta la mitad del argumento de todo el taller: es LA MISMA CLASE.
# No cambiamos el loop. No cambiamos nada de como despachamos tool_calls.
# Heredamos y listo.
#
# Lo unico que agregamos es que se acuerde de que tools uso, para poder mostrarlo
# despues. Eso es instrumentacion nuestra, no es parte del agente.
#
# Que quede clarisimo: si esto falla, no falla el framework (no hay framework) ni
# falla el loop (es el mismo que ya vimos andar). Falla el catalogo de tools.

class AgenteVuelosMuchasTools(AgenteVuelos):
    """El mismo agente de antes, pero con seis tools en vez de una."""

    def __init__(self, tools: dict, system: str, model: str = MODEL):
        super().__init__(tools=tools, system=system, model=model)
        # Lista donde vamos anotando cada tool que el modelo pide, en orden.
        # Sin esto la unica forma de saber que eligio es leerle la mente.
        self.tools_usadas: list[str] = []

    def _tool_schemas(self) -> list[dict]:
        """Los schemas de las seis tools, generados desde los docstrings."""
        # PARA CASA: en la seccion anterior vimos que LangChain puede generar el
        # schema leyendo el docstring. Lo reusamos aca para no tipear seis JSON
        # a mano. El punto de esta seccion no es el schema, es la eleccion.
        return [convert_to_openai_tool(tool(f, parse_docstring=True))
                for f in self.tools.values()]

    def run(self, pregunta: str, max_turns: int = 5) -> str:
        """El loop, otra vez. Igual al de antes, con un print de mas."""
        self.messages.append({"role": "user", "content": pregunta})
        turno = 0
        while turno < max_turns:
            turno += 1
            resp = client.chat.completions.create(
                model=self.model,
                # temperature=1.0 es lo que pide el guion. Lo hablamos en la celda
                # de abajo, porque no es lo que vos crees que es.
                temperature=1.0,
                messages=self.messages,
                tools=self._tool_schemas(),
            )
            msg = resp.choices[0].message

            # Sin tool_calls el modelo ya tiene la respuesta final. Salimos.
            if not msg.tool_calls:
                return msg.content

            # Guardamos el mensaje del asistente TAL CUAL vino, con los tool_calls
            # adentro. Si no lo guardas, la API despues rechaza los resultados
            # porque no sabe a que llamada corresponden.
            self.messages.append(msg.model_dump(exclude_none=True))

            for call in msg.tool_calls:
                nombre = call.function.name
                args = json.loads(call.function.arguments)

                self.tools_usadas.append(nombre)          # para mostrarlo despues
                print(f"  el agente pidio: {nombre}({args})")

                # ACA despachamos nosotros. El modelo no ejecuta nada.
                resultado = self.tools[nombre](**args)

                self.messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    # Recortamos a 3 resultados: el modelo no necesita 5 y ademas
                    # nos ahorra tokens en vivo.
                    "content": json.dumps(resultado[:3], ensure_ascii=False),
                })
        return "Me quede sin turnos."


# El system prompt. No dice "portate mal". Dice algo que diria cualquier PM:
# "priorizá lo mas barato". Eso es todo lo que hace falta.
SYSTEM_CONFUNDIDO = (
    "Sos un asistente de viajes obsesionado con el precio. "
    "Tenes muchas tools parecidas. "
    "Siempre que puedas, prioriza las opciones mas economicas."
)

agente_confundido = AgenteVuelosMuchasTools(
    tools=TOOLS_MUCHAS,
    system=SYSTEM_CONFUNDIDO,
)

print("Agente armado con 6 tools. Todavia no corrio nada.")


### Una aclaracion sobre `temperature`, porque te la van a preguntar

En el loop pusimos `temperature=1.0`. Suena a que estamos subiendo el caos para que falle.
No es asi, y conviene decirlo:

**1.0 es el valor por defecto de la API.** Si no pasas `temperature`, la API usa 1.0. O sea
que no lo subimos: simplemente no lo bajamos. Lo escribimos explicito para que lo veas, no
para trampearlo.

**Y no es lo que hace fallar este demo.** Probamos el mismo pedido con `temperature=0.0` y
el agente elige exactamente las mismas tools equivocadas. La temperatura aca no mueve el
amperimetro.

Entonces `temperature` no afecta la eleccion de tools? Si afecta, en general: el nombre de
la tool sale del modelo como cualquier otro token, asi que pasa por el mismo muestreo. Con
la temperatura muy alta el agente se vuelve erratico y empieza a olvidarse patas del
pedido. Por eso en produccion, si te importa que el ruteo sea predecible, lo pinchas en 0.

Pero el bug de esta celda no es la temperatura. Es el catalogo. Acordate de esto cuando lo
veas elegir.


In [ ]:
# ===========================================================================
# EL PEDIDO COMPLETO, EN VIVO (NARRAR)
# ===========================================================================
# El pedido de siempre, el que arrastramos desde la primera celda del notebook.
PEDIDO = "Buscame vuelos a Bariloche, un hotel cerca del centro y que hacer el finde"

# Lo que NOSOTROS esperabamos que usara. Esto no lo sabe el agente: es nuestro
# contrato mental, el que tenemos en la cabeza cuando escribimos el codigo.
TOOLS_ESPERADAS = {"buscar_vuelos", "buscar_hoteles", "buscar_actividades"}

print(f"PEDIDO: {PEDIDO}\n")
respuesta_confundida = agente_confundido.run(PEDIDO)

# Ahora la parte importante: comparar lo que esperabamos contra lo que uso.
# Sin esta comparacion la respuesta parece impecable y no te enteras de nada.
print("\n" + "=" * 60)
print("ESPERABAMOS: ", sorted(TOOLS_ESPERADAS))
print("USO:         ", agente_confundido.tools_usadas)
print("=" * 60)
for nombre in agente_confundido.tools_usadas:
    marca = "OK " if nombre in TOOLS_ESPERADAS else "MAL"
    print(f"  [{marca}]  {nombre}")
print("=" * 60)

print("\nLA RESPUESTA AL USUARIO:\n")
print(respuesta_confundida[:700])


### Lo que acaba de pasar, congelado

<!-- DIAGRAM: sequenceDiagram. Usuario pide vuelos+hotel+actividades al Agente. El agente elige buscar_vuelos_baratos (no buscar_vuelos), buscar_alojamiento (no buscar_hoteles) y buscar_actividades (esta bien). Marcar las dos primeras como eleccion equivocada. Ultimo mensaje: respuesta prolija al usuario, sin ningun error visible. -->


In [ ]:
# Congelamos el misruteo, porque la salida de arriba se va a ir de pantalla
# en cuanto sigamos hablando.
mostrar_diagrama("muchas-tools-misruteo")


Mira la secuencia y fijate donde esta el problema. No esta en el loop: el loop hizo todo
bien, pidio tres tools y despacho tres tools. El problema es **cuales** tres.

Pedimos vuelos y fue a `buscar_vuelos_baratos`. Pedimos hotel y fue a `buscar_alojamiento`.
Las dos gemelas. Por que? Porque el system prompt dijo "priorizá el precio", y de cada par,
la gemela es la que tiene escrito "con los mejores precios" en el docstring. El modelo leyo
los docstrings y le hizo caso al prompt. **Hizo exactamente lo que le pedimos.**

Y aca esta la parte que da miedo: **la respuesta al usuario esta bien.** Vuelos, hotel,
actividades, todo lindo. Ningun error, ninguna excepcion, ningun log en rojo. Si esto pasa
en produccion no te enteras por un stack trace. Te enteras tres semanas despues, cuando
alguien pregunta por que los precios no coinciden con el otro endpoint.


### 🎤 Te estoy forzando la mano, y te lo digo

Sinceremonos, porque si no te lo digo yo te lo va a preguntar alguien:

> **"Les estoy forzando la mano para ver en 10 segundos lo que en un sistema real les pasa
> en la semana tres."**

Este demo esta armado para fallar. Los docstrings de las gemelas los escribi para que se
pisen, y el system prompt le da al modelo justo la excusa que necesita para preferir la
gemela. Con seis tools bien escritas, `gpt-4o-mini` elige bien casi siempre: lo probamos, y
con las descripciones limpias acierta las tres patas.

El numero real, para que lo tengas: la degradacion seria por cantidad de tools arranca
alrededor de **15 a 20 tools**, no seis. Con catalogos grandes de verdad la cosa se pone
fea rapido: hay mediciones de precision de seleccion cayendo a 13% con catalogos grandes, y
de 43% a 2% al pasar de 4 tools en un dominio a 51 tools en siete dominios.

Entonces este demo no te prueba que seis tools rompen un agente. Te muestra **que forma
tiene** el problema cuando aparece, y te lo muestra ahora en vez de en la semana tres. La
forma es siempre esta: dos tools que se pisan, un prompt razonable, y una respuesta
correcta construida con las tools equivocadas.

Y si te llevas una sola frase de esta seccion, que sea esta: **lo que rompe el ruteo son
los limites que se pisan, no la cantidad de tools.** Dos tools ambiguas te arruinan el dia;
veinte tools bien separadas no. Por eso lo que viene no es "tener menos tools", es "tener
limites claros".

### Entonces, que arreglamos?

Fijate que NO vamos a tocar:

- El loop esta perfecto. Es el mismo de antes y funciono las dos veces.
- No hay framework para culpar. Esto es Python nuestro.
- Bajar `temperature` no lo arregla. Ya lo medimos.

Lo que esta mal es el catalogo: un solo agente con tools que se pisan. **Vamos a partirlo.**


In [ ]:
# ===========================================================================
# MINI EJERCICIOS (opcionales, para despues)
# ===========================================================================
# Las soluciones estan todas al final del notebook.

# EJERCICIO 7 (opcional, para despues)
# Arreglalo por el lado de las descripciones, sin partir el agente.
# Reescribi los docstrings de buscar_vuelos_baratos y buscar_alojamiento para que el
# limite con sus gemelas sea una pared: deci cuando SI y cuando NO usar cada una.
# Volve a correr el agente y mira si cambia lo que elige.
# Pista: el consejo que se repite en todas las guias de diseno de tools es que si dos
# tools pueden contestar el mismo pedido, o las unificas o reescribis las dos hasta
# que el limite sea inequivoco.
# Solucion al final del notebook.

# EJERCICIO 8 (opcional, para despues)
# Comprobalo vos: cambia temperature=1.0 por temperature=0.0 y corre de nuevo el
# mismo pedido. Vas a ver que elige las mismas tools equivocadas.
# Escribi en un comentario, en una linea, por que bajar la temperatura no alcanza.
# Solucion al final del notebook.


## Seccion 5: Lo partimos: un agente por capacidad

Una funcionalidad, un agente, con sus propias tools.

<!-- /build-notebook llena esta seccion desde su plan. -->


## Seccion 6: Quien le contesta al usuario

Los dos agentes hicieron su trabajo y nadie se queda con la respuesta. Ahi entra el supervisor.

<!-- /build-notebook llena esta seccion desde su plan. -->


## Seccion 7: MCP, sin misticismo

Llega ultimo y a proposito: MCP es un protocolo para exponer tools.

<!-- /build-notebook llena esta seccion desde su plan. -->


## Seccion 8: Y si se organizan solos

La pregunta de al lado: y si los agentes se pasan el trabajo entre ellos?

<!-- /build-notebook llena esta seccion desde su plan. -->


## Seccion 9: Cierre

Que construimos, y con que te vas.

<!-- /build-notebook llena esta seccion desde su plan. -->


## Soluciones de los mini ejercicios

Todas las soluciones, en orden. Nada de esto hace falta para seguir el taller.

<!-- /build-notebook llena esta seccion desde su plan. -->


### EJERCICIO 1: agregarle `buscar_hoteles` al agente

Dos cambios: un schema nuevo y una clave mas en el dict `tools`. Lo interesante esta en
la salida: el modelo pide **las dos tools en el mismo turno**, no una por turno. Por eso
el `for call in msg.tool_calls` del loop no era decoracion.


In [ ]:
# SOLUCION EJERCICIO 1: el agente con las dos tools
ESQUEMA_BUSCAR_HOTELES = {
    "type": "function",
    "function": {
        "name": "buscar_hoteles",
        "description": "Busca hoteles en una ciudad argentina.",
        "parameters": {
            "type": "object",
            "properties": {
                "ciudad": {"type": "string", "description": "por ejemplo Bariloche"},
                "zona": {"type": "string",
                         "description": "barrio o area, por defecto centro"},
            },
            # Solo `ciudad` es obligatoria: si el modelo no manda `zona`, el default
            # de Python ("centro") se aplica solo.
            "required": ["ciudad"],
        },
    },
}


class AgenteCompleto(AgenteVuelos):
    def _tool_schemas(self) -> list[dict]:
        return [ESQUEMA_BUSCAR_VUELOS, ESQUEMA_BUSCAR_HOTELES]


agente_completo = AgenteCompleto(
    tools={"buscar_vuelos": buscar_vuelos, "buscar_hoteles": buscar_hoteles},
    system="Sos un agente de viajes argentino. Respondes corto.",
)
print(agente_completo.run("Buscame vuelos de Buenos Aires a Bariloche y un hotel "
                          "cerca del centro."))
# Fijate en la salida: el modelo pide las DOS tools en el turno 1, no una por turno.
print("Roles:", [m["role"] for m in agente_completo.messages])


### EJERCICIO 2: leer la conversacion completa

`self.messages` es una lista de Python comun. Cinco mensajes, y cada uno es un paso del
diagrama. Ojo con el mensaje del assistant: paso por `model_dump`, asi que es un dict y
las `tool_calls` se leen con corchetes.


In [ ]:
# SOLUCION EJERCICIO 2: la conversacion, mensaje por mensaje
for i, m in enumerate(agente_vuelos.messages):
    rol = m["role"]
    if rol == "tool":
        # La respuesta de la tool es el JSON que le devolvimos, como string.
        detalle = f"respuesta de la tool ({len(m['content'])} caracteres)"
    elif m.get("tool_calls"):
        # Este mensaje paso por model_dump, asi que es un DICT: se lee con corchetes.
        detalle = "pidio: " + ", ".join(tc["function"]["name"] for tc in m["tool_calls"])
    else:
        detalle = (m.get("content") or "")[:60].replace("\n", " ")
    print(f"{i}. {rol:10s} {detalle}")


### EJERCICIO 3: el freno de mano

Con `max_turns=1` el modelo gasta el unico turno pidiendo la tool, asi que nunca llega a
escribir la respuesta final. `run()` devuelve `"Me quede sin turnos."` y los roles quedan
cortados justo antes del ultimo `assistant`. Para eso existe el techo.


In [ ]:
# SOLUCION EJERCICIO 3: que pasa cuando se agotan los turnos
agente_corto = AgenteVuelos(
    tools={"buscar_vuelos": buscar_vuelos},
    system="Sos un agente de viajes argentino. Respondes corto.",
)
print(agente_corto.run("Buscame vuelos de Buenos Aires a Bariloche.", max_turns=1))
# Devuelve "Me quede sin turnos.": el modelo gasto el unico turno pidiendo la tool,
# asi que nunca llego a escribir la respuesta final.
print("Roles:", [m["role"] for m in agente_corto.messages])
# Roles queda en ['system', 'user', 'assistant', 'tool']: la historia cortada al medio,
# justo antes del ultimo assistant. Por eso `run()` devuelve un string igual, siempre.


### EJERCICIO 4: el mismo agente con otro `system_prompt`

Cambias un solo parametro y la respuesta cambia entera. El prompt es parte del agente.


In [ ]:
# SOLUCION EJERCICIO 4: otro system_prompt, misma tool
agente_una_linea = create_agent(
    model=f"openai:{MODEL}",
    tools=[vuelos_tool],                       # la misma tool de siempre
    system_prompt="Contesta en una sola linea, sin listas.",
)
estado_corto = agente_una_linea.invoke(
    {"messages": [{"role": "user", "content": PREGUNTA_VUELOS}]}
)
print(estado_corto["messages"][-1].content)


### EJERCICIO 5: el grafo, dibujado por LangGraph

`draw_mermaid()` te imprime el grafo sin instalar nada. Buscá la ultima flecha:
`tools -.-> model`. Esa es el `while` que escribimos a mano.

Ojo con un detalle: la flecha es **punteada** (`-.->`), no solida. En Mermaid la
punteada marca una arista condicional, que es justo lo que era nuestro `if`.


In [ ]:
# SOLUCION EJERCICIO 5: imprimir el grafo
diagrama = agente_vuelos_lg.get_graph().draw_mermaid()
print(diagrama)

# La flecha que nos importa. Fijate que es punteada: es una arista condicional.
print("la flecha del loop esta?:", "tools -.-> model" in diagrama)


### EJERCICIO 6: que pasa sin `parse_docstring=True`

El mas interesante de los tres. Sin el flag, LangChain **no** adivina: te mete el
docstring entero, con el bloque `Args:` y todo, dentro de un solo campo `description`,
y las descripciones por argumento quedan en `None`.

El modelo lee esas descripciones por argumento para decidir que mandar. Perderlas no
rompe nada de forma visible, y esa es exactamente la clase de detalle que despues te
cuesta una tarde de debugging.


In [ ]:
# SOLUCION EJERCICIO 6: el mismo docstring, parseado mal a proposito
sin_flag = tool(buscar_vuelos)                    # sin parse_docstring=True
esquema_sin = convert_to_openai_tool(sin_flag)
esquema_con = convert_to_openai_tool(vuelos_tool)  # el de la celda de arriba

print("=== SIN parse_docstring=True ===")
print(json.dumps(esquema_sin, indent=2, ensure_ascii=False))
print()

# La diferencia que importa, lado a lado.
props_sin = esquema_sin["function"]["parameters"]["properties"]
props_con = esquema_con["function"]["parameters"]["properties"]
print("descripciones por argumento SIN el flag:",
      {k: v.get("description") for k, v in props_sin.items()})
print("descripciones por argumento CON el flag:",
      {k: v.get("description") for k, v in props_con.items()})


### EJERCICIO 7: arreglarlo escribiendo mejor los docstrings

No hace falta partir el agente para mejorar el ruteo: alcanza con que el limite entre las
gemelas sea explicito. Fijate que ahora cada docstring dice cuando **no** usarse.

Esto mejora mucho el ruteo, y sigue siendo fragil: depende de que cada persona que agregue
una tool escriba el limite bien. Por eso la solucion de fondo es la de la seccion que viene.


In [ ]:
# SOLUCION EJERCICIO 7: limites explicitos en los docstrings
def buscar_vuelos_baratos_v2(origen: str, destino: str) -> list[dict]:
    """Busca SOLO ofertas y promociones de ultimo momento en vuelos.

    No usar para una busqueda normal de vuelos: para eso usa buscar_vuelos.
    Usar unicamente si el usuario pide explicitamente ofertas o descuentos.

    Args:
        origen: ciudad de salida, por ejemplo "Buenos Aires"
        destino: ciudad de llegada, por ejemplo "Bariloche"
    """
    return _normalizar(_serper(f"ofertas vuelos {origen} {destino}"))


def buscar_alojamiento_v2(ciudad: str, zona: str = "centro") -> list[dict]:
    """Busca SOLO hostels y departamentos temporarios, no hoteles.

    No usar para hoteles: para eso usa buscar_hoteles.

    Args:
        ciudad: por ejemplo "Bariloche"
        zona: barrio o area, por defecto "centro"
    """
    return _normalizar(_serper(f"hostel departamento temporario {zona} {ciudad}"))


agente_arreglado = AgenteVuelosMuchasTools(
    tools={
        "buscar_vuelos": buscar_vuelos,
        "buscar_vuelos_baratos": buscar_vuelos_baratos_v2,   # con el limite claro
        "buscar_hoteles": buscar_hoteles,
        "buscar_alojamiento": buscar_alojamiento_v2,         # con el limite claro
        "buscar_transporte": buscar_transporte,
        "buscar_actividades": buscar_actividades,
    },
    system=SYSTEM_CONFUNDIDO,     # el MISMO prompt de antes, no lo tocamos
)
agente_arreglado.run(PEDIDO)
print()
print("ESPERABAMOS:", sorted(TOOLS_ESPERADAS))
print("USO:        ", agente_arreglado.tools_usadas)


### EJERCICIO 8: por que bajar `temperature` no alcanza

Porque la temperatura cambia **cuanto** varia la eleccion, no **cual** es la mejor
candidata. Con las descripciones pisadas, la gemela es la opcion mas probable para el
modelo, y a temperatura 0 lo que hacemos es elegir la mas probable siempre: la equivocada,
pero de forma consistente.


In [ ]:
# SOLUCION EJERCICIO 8: temperature=0 no lo arregla
class AgenteFrio(AgenteVuelosMuchasTools):
    """El mismo agente confundido, pero con temperature=0."""

    def run(self, pregunta: str, max_turns: int = 5) -> str:
        # Reusamos todo el loop del padre y solo cambiamos la temperatura,
        # parcheando el cliente por un momento. Mas simple: copiar el metodo.
        self.messages.append({"role": "user", "content": pregunta})
        turno = 0
        while turno < max_turns:
            turno += 1
            resp = client.chat.completions.create(
                model=self.model,
                temperature=0.0,             # <-- el unico cambio
                messages=self.messages,
                tools=self._tool_schemas(),
            )
            msg = resp.choices[0].message
            if not msg.tool_calls:
                return msg.content
            self.messages.append(msg.model_dump(exclude_none=True))
            for call in msg.tool_calls:
                nombre = call.function.name
                args = json.loads(call.function.arguments)
                self.tools_usadas.append(nombre)
                resultado = self.tools[nombre](**args)
                self.messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": json.dumps(resultado[:3], ensure_ascii=False),
                })
        return "Me quede sin turnos."


agente_frio = AgenteFrio(tools=TOOLS_MUCHAS, system=SYSTEM_CONFUNDIDO)
agente_frio.run(PEDIDO)
print("USO con temperature=0:", agente_frio.tools_usadas)
# Las mismas gemelas. La temperatura no arregla un catalogo ambiguo:
# solo hace que elijas la opcion equivocada de forma mas consistente.
